In [ ]:
##DATA INGESTION

In [1]:
###document datastructure
from langchain_core.documents import Document

In [ ]:
doc=Document(
    page_content="This is a sample document.",  
    metadata={
        "source": "sample_source",
        "author": "John Doe",
        "pages": 10
    }
)

In [3]:
##create a simplee txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts={
    "../data/text_files/sample1.txt": """Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and active community contribute to its popularity in various fields such as web development, data analysis, artificial intelligence, and scientific computing. key features of Python include dynamic typing, automatic memory management, and a large ecosystem of third-party packages. Its versatility and ease of use make it a preferred choice for both beginners and experienced developers.""",
}

for filepath,content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

In [5]:
sample_texts={
    "../data/text_files/sample2.txt": """machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. It involves training models on large datasets to recognize patterns, extract insights, and improve performance over time without being explicitly programmed for specific tasks. Machine learning techniques are widely used in various applications, including image and speech recognition, natural language processing, recommendation systems, and autonomous vehicles. Key concepts in machine learning include supervised learning, unsupervised learning, reinforcement learning, and deep learning.""",
}

for filepath,content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)

In [4]:
from langchain_community.document_loaders import TextLoader

In [ ]:
loader=TextLoader("../data/text_files/sample1.txt")
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and active community contribute to its popularity in various fields such as web development, data analysis, artificial intelligence, and scientific computing. key features of Python include dynamic typing, automatic memory management, and a large ecosystem of third-party packages. Its versatility and ease of use make it a preferred choice for both beginners and experienced developers.")]


: 

In [11]:
### directory datastructure
from langchain_community.document_loaders import DirectoryLoader

dir_loader=DirectoryLoader("../data/text_files",
                            glob="**/*.txt", ##Pattern to match files in the directory
                            loader_cls=TextLoader, ##LOADER CLASS TO USE FOR EACH FILE
                            show_progress=False,
                            loader_kwargs={"encoding":"utf-8"}

)
dir_documents=dir_loader.load() 
print(dir_documents)
print(type(dir_documents))

[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and active community contribute to its popularity in various fields such as web development, data analysis, artificial intelligence, and scientific computing. key features of Python include dynamic typing, automatic memory management, and a large ecosystem of third-party packages. Its versatility and ease of use make it a preferred choice for both beginners and experienced developers."), Document(metadata={'source': '..\\data\\text_files\\sample2.txt'}, page_content='machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions 

### pdf loader

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, PyMuPDFLoder

dir_loader=DirectoryLoader("../data/pdf",
                            glob="**/*.pdf", ##Pattern to match files in the directory
                            loader_cls=PyMuPDFLoader, ##LOADER CLASS TO USE FOR EACH FILE
                            show_progress=False,

)
pdf_documents=dir_loader.load() 
print(pdf_documents)
print(type(pdf_documents))

ImportError: cannot import name 'PyMuPDFLoader' from 'langchain_core.document_loaders' (c:\Users\gargi\Documents\rag\.venv\Lib\site-packages\langchain_core\document_loaders\__init__.py)

## embedding and vector store db

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer  ##embeddings will be avail inside this package
import chromadb ##setup a vector database to store embeddings
from chromadb.config import Settings
import uuid ##generate unique ids for each document
from typing import List,Dict,Any,Tuple 
from sklearn.metrics.pairwise import cosine_similarity ##compute similarity between embeddings


c:\Users\gargi\Documents\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: DLL load failed while importing _mt19937: An Application Control policy has blocked this file.

In [2]:
class EmbeddingManager:
    """handles documeny embedding generation usinng sentence-transformers and storage in ChromaDB"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        
        """
        initializes the EmbeddingManager with a specified sentence-transformers model for embedding generation.
        Args:
            model_name (str): Name of the sentence-transformers model to use for embedding generation.
        """
        self.model = model_name
        self.model =None
        self._load_model()

    def _load_model(self):
        """loads the sentence-transformers model for embedding generation."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding Dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  
        """
        generates embeddings for a list of texts using the loaded sentence-transformers model.
        Args:
            texts (List[str]): List of texts to generate embeddings for.
            Returns:
                np.ndarray: Array of generated embeddings.
                """
        if self.model is None:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")
        try:
            embeddings = self.model.encode(texts, convert_to_numpy=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

    def get_embedding_dimension(self) -> int:
        """
        retrieves the dimension of the embeddings generated by the model.
        Returns:
            int: Dimension of the embeddings.
        """
        if self.model is None:
            raise ValueError("Model is not loaded. Please load the model before retrieving embedding dimension.")
        return self.model.get_sentence_embedding_dimension()




###initialize the EmbeddingManager with a specific model
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager

Error loading model: 'EmbeddingManager' object has no attribute 'model_name'


AttributeError: 'EmbeddingManager' object has no attribute 'model_name'

### VECTORE STORE

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector database. 
      """
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initializes the VectorStore with a specified collection name and persistence directory.
        Args:
            collection_name (str): Name of the ChromaDB collection to store embeddings.
            persist_directory (str): Directory path to persist the ChromaDB database.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() ###this function initializes the vector database and creates a collection for storing document embeddings.


    def _initialize_store(self):
        """Initializes the ChromaDB client and creates a collection for storing document embeddings."""
        try:
            ##create a persistent chroma db
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            ) ##creates a client whicch has a reference to the persistent database stored in the specified directory or chromadb vectore database
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Document Embeddings for Retrieval-Augmented Generation (RAG)"},
            ) ##place to store the embeddings of the documents in the collection
            print(f"Vector store initialized with collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embedings: np.ndarray):
        """
        Adds documents and their corresponding embeddings to the ChromaDB collection.
        Args:
            documents (List[Any]): List of documents to add to the collection.
            embeddings (np.ndarray): Array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")    
        print(f"Adding {len(documents)} documents to the vector store...")

        #prepare the data for insertion into the collection
        ids = [] 
        metadatas = [] 
        documents_content = []
        embeddings = []   ##convert numpy array to list of lists


        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = str(uuid.uuid4())  ##generate a unique id for each document
            ids.append(doc_id)

            ##prepare metadate
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i  ##add the index of the document in the list to the metadata
            metadata['content_length'] = len(doc.page_content)  ##add the length of the document content to the metadata
            metadatas.append(metadata)


            ##document content
            documents_content.append(doc.page_content)

            ##Embedding
            embeddings.append(embedding.tolist())  ##convert numpy array to list of lists

        ##add the documents, metadatas, and embeddings to the collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_content,
                embeddings=embeddings
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")    
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore


    
    

    

_IncompleteInputError: incomplete input (1808714509.py, line 1)

In [ ]:
chunks

In [ ]:
###convert  the chunks into embeddings and store them in the vector database
texts=[doc.page_content for doc in chunks]
texts

##generate embeddings for the document chunks
embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks, embeddings)

### RAG RETRIVER PIIPELINE FOR VECTORRSTORE

In [ ]:
class RAGRetriever:
    """Handles quuery-based retrieval of relevant document chunks from the vector database using embeddings and cosine similarity."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initializes the RAGRetriever with a specified vector store and embedding manager.
        Args:
            vector_store (VectorStore): Instance of the VectorStore class for managing document embeddings.
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager class for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

 ##retriver is build upon vectorestore and it is nothing but whatever query we pass it will generate embedding for the query and then it will search in the vectorstore for the most similar document chunks and return them
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieves the most relevant document chunks from the vector store based on the query.
        Args:
            query (str): The query string to retrieve relevant documents for.
            top_k (int): The number of top relevant documents to retrieve.
        Returns:

            List[Dict[str, Any]]: List of dictionaries containing the retrieved document chunks and their metadata.
        """
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Retrieve all embeddings and metadata from the vector store
        all_embeddings = np.array(self.vector_store.collection.get_all_embeddings())
        all_metadatas = self.vector_store.collection.get_all_metadatas()
        all_documents = self.vector_store.collection.get_all_documents()

        ##search in vectorstore for the most similar document chunks and return them
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #preocess the results to include document content and metadata

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distance to similarity score
                    similarity_score = 1 - distance  ##cosine similarity is 1-distance

                    if similarity_score < 0:
                        similarity_score = 0.0

                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score
                    })

            print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")

            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise


        rag_retriever=RAGRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.query("What is Artificial Intelligence?", top_k=3)